# Loss Functions in Deep Learning

## 1. What is a Loss Function?

A **loss function** is a way of evaluating how well an algorithm models a given dataset — it measures the performance of your algorithm.

- If the loss value is **large** → the algorithm is performing **poorly**
- If the loss value is **small** → the algorithm is performing **well**
- A loss function is just a mathematical function — nothing more exotic than that
- It's a function of the model's **parameters**. As you adjust the parameters, the loss value changes

**Example — Simple Linear Regression:**
`y = mx + b`

- Changing `m`, `b`, or both changes the loss value
- Goal: find the `m, b` combination that **minimizes** the loss
- This is done using an algorithm like **Gradient Descent**

---

## 2. Why is the Loss Function Important?

> "You can't improve what you can't measure" — this is exactly why loss functions matter.

The loss function acts as **feedback** — it tells you how wrong your predictions are, so you know which direction to adjust your parameters.

**Typical ML training loop:**
1. Model starts with random parameters (e.g., a random line)
2. Calculate the loss for the current parameters
3. Adjust parameters (e.g., rotate the line) based on the loss, using Gradient Descent
4. Recalculate loss with new parameters
5. Repeat until loss is minimized → model is "trained"

> The loss function is essentially the **"eyes" of the algorithm** — it tells the algorithm where to go and what parameter values to set. Every ML/DL algorithm (regression, classification, etc.) depends on some loss function.

---

## 3. How Loss Functions Work in Deep Learning

**Example dataset:** Student `CGPA`, `IQ` → `Package (LPA)`

1. Initialize weights and biases randomly
2. Take one data point, run **forward propagation** through the network
3. Get a prediction (e.g., `3.79`)
4. Pick a loss function (e.g., Mean Squared Error for regression)
5. Calculate the loss — tells you how good/bad this prediction was
6. Use **Gradient Descent** to update weights and biases
7. Move to the next data point, repeat forward prop → loss → weight update
8. Repeat over the full dataset, across multiple **epochs**, until loss is minimized

This confirms: loss functions work exactly the same way in deep learning as they do in classic ML.

---

## 4. Types of Loss Functions (Overview)

### Regression
- Mean Squared Error (MSE)
- Mean Absolute Error (MAE)
- Huber Loss

### Classification
- Binary Cross Entropy
- Categorical Cross Entropy
- Sparse Categorical Cross Entropy
- Hinge Loss (used in SVMs)

### Specialized architectures
- **Autoencoders** → KL Divergence
- **GANs** → Discriminator loss / Minimax loss
- **Object Detection** → Focal Loss
- **Embeddings** → Triplet Loss

> You can also define **custom loss functions** for research purposes (frameworks like Keras support this). Choice of loss function depends entirely on the problem — each has its own trade-offs.

---

## 5. Loss Function vs. Cost Function

These are often used interchangeably, but technically they're different (good interview point):

| Term | Applies to |
|---|---|
| **Loss function** | A **single** training example |
| **Cost function** | The **entire** training set (average of all individual losses) |

**Example:**
- Loss (single point): `(y − ŷ)²`
- Cost (all `n` points): `(1/n) Σ (y − ŷ)²`

---

## 6. Mean Squared Error (MSE)
*(also called Squared Loss or L2 Loss)*

**Formula:**
- Loss (single point): `(y − ŷ)²`
- Cost (all points): `(1/n) Σ (y − ŷ)²`

**Usage:** Regression problems. The **output neuron's activation must be linear** to use MSE.

**Why squared and not just the raw difference?**
- Some `(y − ŷ)` values are positive, some negative
- If summed directly without squaring, positive and negative errors **cancel out**, understating total error
- Squaring makes everything positive so every point contributes correctly

**Quadratic nature (key property):**
- Because of squaring, error is **magnified** for points further from the true value
- `|error| = 1` → contributes `1` to loss
- `|error| = 2` → contributes `4` to loss (not just double — quadratic growth)
- Points far from the true value dominate the weight updates; nearby points barely move the model

**Advantages:**
1. Easy to interpret
2. Always differentiable → gradient descent works smoothly
3. Only **one local minimum** (convex) → guaranteed path to the global minimum, no risk of getting stuck

**Disadvantages:**
1. Output unit is **squared** (e.g., if target is in LPA, loss is in LPA²) — needs a square root to interpret meaningfully (this is where RMSE comes from)
2. **Not robust to outliers** — a single extreme point can drag the model badly, hurting overall fit

**Deep learning setup:**
- Output layer activation → **Linear**
- Model compile: `loss = 'mean_squared_error'`

---

## 7. Mean Absolute Error (MAE)
*(also called L1 Loss)*

**Formula:**
- Loss (single point): `|y − ŷ|`
- Cost (all points): `(1/n) Σ |y − ŷ|`

**Advantages:**
1. Intuitive and easy to understand
2. Same **unit** as the target variable — no square-root conversion needed
3. Doesn't over-penalize outliers (no quadratic blow-up) — good choice when your data has outliers

**Disadvantages:**
1. The loss curve is **not differentiable at zero** (there's a sharp kink) — gradient descent needs **sub-gradients** here, which is a bit more computationally expensive

**Rule of thumb:** No outliers → use MSE. Significant outliers → use MAE.

---

## 8. Huber Loss

**Motivation:** Combines the strengths of MSE and MAE.
- MSE over-reacts to outliers (quadratic penalty)
- MAE treats outliers exactly like normal points (no special handling)

If, say, 25% of your points look like "outliers," are they really outliers, or just a different sub-population? You want a fit that balances between fully chasing the majority (MSE-style) and ignoring the minority group (MAE-style).

**How it works:**
- Behaves like **MSE** for points that are close to the prediction (non-outliers)
- Behaves like **MAE** for points that are far off (likely outliers)
- Has a tunable parameter, often called **delta (δ)**, that decides the threshold between the two behaviors — you tune this to control model performance

**Best suited for:** Data with a genuine mix of outliers and normal points.

---

## 9. Binary Cross Entropy
*(also called Log Loss)*

**Use case:** Binary classification (exactly two classes — Yes/No). Same loss function used in Logistic Regression.

**Formula (single point):**
```
Loss = −[ y·log(ŷ) + (1−y)·log(1−ŷ) ]
```
where `y` = true label, `ŷ` = predicted probability from the network.

**Cost function (all `m` points):**
```
Cost = −(1/m) Σ [ yᵢ·log(ŷᵢ) + (1−yᵢ)·log(1−ŷᵢ) ]
```

**Architecture requirement:**
- Output layer → exactly **one neuron**
- That neuron's activation → must be **Sigmoid** (hidden layers can use ReLU, sigmoid, etc. — doesn't matter)

**Worked example from the lecture:**
- Student A: prediction `ŷ = 0.738`, true label `y = 1` → Loss ≈ `−log(0.738) ≈ 0.1`
- Student B: prediction `ŷ = 0.25`, true label `y = 0` → Loss ≈ `−log(1 − 0.25) ≈ 0.12`

**Advantages:**
- Differentiable → gradient descent works fine

**Disadvantages:**
- Can have multiple local minima (unlike MSE's single global minimum)
- Less intuitive to interpret just by looking at the formula

*(Full derivation uses Maximum Likelihood Estimation — covered separately in a dedicated Logistic Regression video.)*

---

## 10. Categorical Cross Entropy

**Use case:** Multi-class classification (3+ classes). Example: Placement outcome with classes `Placed / Not Placed / Maybe`.

**Formula (single point):**
```
Loss = − Σⱼ yⱼ · log(ŷⱼ)      for j = 1 to k (k = number of classes)
```

**Architecture requirements:**
- Number of output neurons = number of classes
- Output activation → **Softmax**
- Softmax: `ŷᵢ = e^(zᵢ) / Σⱼ e^(zⱼ)` — outputs are all between 0–1 and sum to 1

**Data prep:** Labels must be **one-hot encoded**
- e.g., `Yes → [1,0,0]`, `No → [0,1,0]`, `Maybe → [0,0,1]`

**Why calculation simplifies:** Since labels are one-hot, every term except the true class's term is multiplied by `0` and disappears. So effectively:
```
Loss = −log(ŷ_true_class)
```
Only the predicted probability assigned to the correct class matters.

**Cost function (all points):**
```
Cost = −(1/n) Σᵢ Σⱼ yᵢⱼ · log(ŷᵢⱼ)
```

---

## 11. Sparse Categorical Cross Entropy

- **Mathematically identical** to Categorical Cross Entropy — no real difference in what's computed
- Only difference is **label encoding**:
  - Categorical Cross Entropy → needs **one-hot encoded** labels
  - Sparse Categorical Cross Entropy → uses plain **integer-encoded** labels (e.g., `0–9` for MNIST digits, no one-hot needed)

**Advantage:** Slightly **faster** — no one-hot encoding step, and you only need the log of the true class's predicted probability rather than computing across a full one-hot vector. Recommended when you have a **large number of classes** (e.g., MNIST with 10 classes).

---

## 12. Summary Cheat Sheet

| Problem Type | Recommended Loss Function |
|---|---|
| Regression — no outliers | Mean Squared Error (MSE) |
| Regression — has outliers | Mean Absolute Error (MAE) |
| Regression — mix of outliers & normal points | Huber Loss |
| Binary Classification | Binary Cross Entropy |
| Multi-class Classification (few classes) | Categorical Cross Entropy |
| Multi-class Classification (many classes) | Sparse Categorical Cross Entropy |

---

## Next Topic
Backpropagation in deep learning — how it uses these loss functions during training (mentioned as the next video in the series).